Optimizer comparison in PyTorch

Compare SGD, Momentum, RMSprop, Adam, AdamW on optimization problem




In [9]:
import torch
import torch.optim as optim

# ============================================================
# OPTIMIZER COMPARISON ON ROSENBROCK FUNCTION
# ============================================================
# f(x,y) = (1-x)² + 100(y-x²)²  |  Minimum at (1,1)

def rosenbrock(xy):
  x, y = xy[0], xy[1]
  return (1 - x)**2 + 100 * (y - x**2)**2

# SGD typically needs higher lr than adam
optimizers = {
    'SGD' : lambda p: optim.SGD(p, lr=0.01),
    'SGD+Momentum' : lambda p: optim.SGD(p, lr=0.01, momentum=0.9),
    'RMSprop' : lambda p: optim.RMSprop(p, lr=0.001),
    'ADAM' : lambda p: optim.Adam(p, lr=0.001),
    'ADAMW' : lambda p: optim.AdamW(p, lr=0.001, weight_decay=0.01),
}

print("Optimizers comparison (500) steps")
print('=' * 40)

for name, opt_fn in optimizers.items():
  xy = torch.tensor([-2.0,2.0],requires_grad=True)
  optimizer = opt_fn([xy])

  for _ in range(500):
    optimizer.zero_grad()
    loss = rosenbrock(xy)
    loss.backward()
    optimizer.step()

  print(f"{name:15s} {xy[0]:.3f}, {xy[1]:.3f}, Loss: {rosenbrock(xy):.4f}")

print("=" * 55)
print("Target: (1.000, 1.000), Loss: 0.0000")




Optimizers comparison (500) steps
SGD             nan, nan, Loss: nan
SGD+Momentum    nan, nan, Loss: nan
RMSprop         -1.587, 2.428, Loss: 7.4935
ADAM            -1.665, 2.351, Loss: 24.8643
ADAMW           -1.658, 2.338, Loss: 23.9756
Target: (1.000, 1.000), Loss: 0.0000


AdamW with warmup + cosine decay

Production training setup for transformers (matches the warmup + cosine recommendation)

In [10]:
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

# ============================================================
# MODERN SETUP: AdamW + Warmup + Cosine Decay
# ============================================================
# This matches the "In Practice" recommendation for transformers:
# - AdamW (decoupled weight decay, NOT Adam + L2)
# - Linear warmup (stability in early steps)
# - Cosine decay (smooth convergence


model = torch.nn.Linear(768, 768)

# adam with standard transformers hyperparameters
optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay = 0.01,
    betas = (0.9, 0.99)
)
total_epochs, warmup_epochs = 100, 5

# warmup 1% to 100% of lr over first 5 epochs
warmup = LinearLR(optimizer, start_factor = 0.01, total_iters = warmup_epochs)

# cosine: smooth decay to near zero after warmup
cosine = CosineAnnealingLR(optimizer, T_max = total_epochs - warmup_epochs, eta_min=1e-6)

# combine: warmup first and then cosine
scheduler = SequentialLR(optimizer, [warmup,cosine], milestones= [warmup_epochs])

print("Learning Rate Schedule")
for epoch in range(total_epochs):
  current_lr = optimizer.param_groups[0]['lr']
  if epoch % 20 == 0 or epoch < 6:
    print(f"Epoch: {epoch:3d}: lr={current_lr:.6f}")

  scheduler.step() # advances lr for the next epoch


Learning Rate Schedule
Epoch:   0: lr=0.000010
Epoch:   1: lr=0.000208
Epoch:   2: lr=0.000406
Epoch:   3: lr=0.000604
Epoch:   4: lr=0.000802
Epoch:   5: lr=0.001000
Epoch:  20: lr=0.000940
Epoch:  40: lr=0.000701
Epoch:  60: lr=0.000378
Epoch:  80: lr=0.000106


/usr/local/lib/python3.13/dist-packages/torch/optim/lr_scheduler.py:1195: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
